# Microsoft Agent Framework — the successor of AutoGen + Semantic Kernel

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USER/SA_LIB_WORKSHOP_Day2/blob/main/05_microsoft_agent_framework/05_microsoft_agent_framework.ipynb)

**SA-LIB AI Workshop 2026 — Day 2: AI Agents**

Microsoft's agent framework: a chat client + `Agent` + `agent.run()`. Sessions give memory; workflows (`SequentialBuilder`, `GroupChatBuilder`, `HandoffBuilder`, ...) give multi-agent.

Same example in every framework: **(1) first agent → (2) tools → (3) memory → (4) fallback → (5) multi-agent → complete agent**.

## 0. Setup
Install the packages (takes ~1 minute in Colab).

In [ ]:
%pip install -q "agent-framework>=1.0" "python-dotenv" "ddgs"

### API keys
* **Colab:** click the 🔑 *Secrets* icon on the left, add e.g. `GROQ_API_KEY`, and enable notebook access.
* **VS Code / Jupyter / Linux:** put your keys in the `.env` file in the repo root (copy `.env.example`).
* Otherwise you'll be asked to paste a key below.

Choose the model provider with `LLM_PROVIDER`: `groq` · `gemini` · `openai` · `deepseek` · `openrouter` · `ollama`.

In [ ]:
import os, getpass
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))          # VS Code / Linux: read ../.env
try:                                           # Colab: read the Secrets panel
    from google.colab import userdata
    for key in ['GROQ_API_KEY', 'GOOGLE_API_KEY', 'OPENAI_API_KEY', 'DEEPSEEK_API_KEY', 'OPENROUTER_API_KEY', 'LANGSMITH_API_KEY', 'LANGFUSE_PUBLIC_KEY', 'LANGFUSE_SECRET_KEY', 'LANGFUSE_HOST', 'LLM_PROVIDER', 'LLM_MODEL']:
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
except ImportError:
    pass

os.environ.setdefault("LLM_PROVIDER", "groq")   # <- change the provider here if you like
provider = os.environ["LLM_PROVIDER"]
key_name = {"groq": "GROQ_API_KEY", "gemini": "GOOGLE_API_KEY", "openai": "OPENAI_API_KEY",
            "deepseek": "DEEPSEEK_API_KEY", "openrouter": "OPENROUTER_API_KEY"}.get(provider)
if key_name and not os.getenv(key_name):
    os.environ[key_name] = getpass.getpass(f"{key_name}: ")
print("provider:", provider)

### Shared helper: `workshop_common.py`
Reads the provider settings, and holds the two tools every example uses: `web_search` (free DuckDuckGo) and `word_count`.

In [ ]:
%%writefile workshop_common.py
"""
workshop_common.py — shared helpers for every framework folder in this workshop.

(the SAME file is copied into each folder so every folder works on its own —
 edit shared/workshop_common.py and run `python tools/sync_common.py` to update the copies)

What it gives you:
  get_llm_config()   -> which provider / model / base_url / api key to use (read from .env)
  fallback_order()   -> the list of providers to try, in order, for the fallback examples
  ping()             -> quick "are you alive?" check against a provider
  web_search()       -> free DuckDuckGo search (no key)
  word_count()       -> the tiny "tool you wrote yourself" from the original guide

Every provider below speaks the OpenAI chat-completions protocol, which is why every
framework in this workshop can use every provider: we just change base_url + api_key + model.
"""

from __future__ import annotations

import os
from dataclasses import dataclass

from dotenv import find_dotenv, load_dotenv

# walks up from where you run python, so the .env in the repo root is always found
load_dotenv(find_dotenv(usecwd=True))

PROVIDERS = {
    #  name        base_url                                                     api-key env var       default model
    "groq":       ("https://api.groq.com/openai/v1",                           "GROQ_API_KEY",       "llama-3.3-70b-versatile"),
    "gemini":     ("https://generativelanguage.googleapis.com/v1beta/openai/", "GOOGLE_API_KEY",     "gemini-2.5-flash"),
    "openai":     ("https://api.openai.com/v1",                                "OPENAI_API_KEY",     "gpt-4o-mini"),
    "deepseek":   ("https://api.deepseek.com",                                 "DEEPSEEK_API_KEY",   "deepseek-chat"),
    "openrouter": ("https://openrouter.ai/api/v1",                             "OPENROUTER_API_KEY", "meta-llama/llama-3.3-70b-instruct:free"),
    "ollama":     (os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"),  None,                 "qwen2.5:7b"),
}


@dataclass
class LLMConfig:
    provider: str
    model: str
    base_url: str
    api_key: str

    @property
    def litellm_model(self) -> str:
        """Model name in LiteLLM format (used by CrewAI and Google ADK): 'openai/<model>'
        means 'an OpenAI-compatible endpoint' — combined with base_url it works for every provider."""
        return f"openai/{self.model}"


def get_llm_config(provider: str | None = None) -> LLMConfig:
    """Read LLM_PROVIDER / LLM_MODEL / <PROVIDER>_API_KEY from the environment."""
    provider = (provider or os.getenv("LLM_PROVIDER") or "groq").strip().lower()
    if provider not in PROVIDERS:
        raise ValueError(f"Unknown LLM_PROVIDER '{provider}'. Choose one of: {', '.join(PROVIDERS)}")
    base_url, key_env, default_model = PROVIDERS[provider]
    api_key = os.getenv(key_env) if key_env else "ollama"  # ollama runs locally, no key
    if not api_key:
        raise RuntimeError(f"{key_env} is missing — add it to your .env file (see .env.example).")
    # LLM_MODEL only overrides the model of the MAIN provider, not of the fallbacks
    main_provider = (os.getenv("LLM_PROVIDER") or "groq").strip().lower()
    model = (os.getenv("LLM_MODEL") if provider == main_provider else None) or default_model
    return LLMConfig(provider, model, base_url, api_key)


def fallback_order() -> list[str]:
    """Providers to try, in order. Default: your LLM_PROVIDER first, then LLM_FALLBACKS (groq,gemini)."""
    main = (os.getenv("LLM_PROVIDER") or "groq").strip().lower()
    extra = [p.strip().lower() for p in os.getenv("LLM_FALLBACKS", "groq,gemini").split(",") if p.strip()]
    return list(dict.fromkeys([main, *extra]))  # remove duplicates, keep order


def ping(cfg: LLMConfig) -> None:
    """Send a 1-token request. Raises an exception if the provider is down, rate-limited or the key is wrong."""
    from openai import OpenAI

    OpenAI(base_url=cfg.base_url, api_key=cfg.api_key, timeout=20, max_retries=0).chat.completions.create(
        model=cfg.model, messages=[{"role": "user", "content": "ping"}], max_tokens=1
    )


def web_search(query: str) -> str:
    """Search the web with DuckDuckGo (free, no key) and return the top results as text."""
    try:
        from ddgs import DDGS

        results = DDGS().text(query, max_results=3)
        return "\n".join(f"- {r['title']}: {r['body']} ({r['href']})" for r in results) or "no results"
    except Exception as e:  # the free search is rate-limited: return the error instead of crashing
        return f"web search failed ({e}). try again in a few seconds."


def word_count(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())

## 1. `01_first_agent.py`

Microsoft Agent Framework 01 — your first agent
Agent Framework is Microsoft's successor to AutoGen + Semantic Kernel.
pieces: a CHAT CLIENT (which model) + an Agent (instructions + tools) + agent.run(...)

In [ ]:
import asyncio

from agent_framework import Agent
from agent_framework.openai import OpenAIChatCompletionClient

from workshop_common import get_llm_config

cfg = get_llm_config()
# OpenAIChatCompletionClient speaks the chat-completions protocol -> works with every provider in .env
client = OpenAIChatCompletionClient(model=cfg.model, api_key=cfg.api_key, base_url=cfg.base_url)

agent = Agent(client, instructions="You are a helpful assistant. Be concise and accurate.", name="assistant")


async def main():
    result = await agent.run("explain what an AI agent is in two sentences.")
    print(f"[{cfg.provider}:{cfg.model}]", result.text)


if __name__ == "__main__":
    await main()

## 2. `02_agent_with_tools.py`

Microsoft Agent Framework 02 — giving your agent tools
plain python functions are tools. the @tool decorator is optional — use it to rename or describe a tool.

In [ ]:
import asyncio

from agent_framework import Agent, tool
from agent_framework.openai import OpenAIChatCompletionClient

import workshop_common as wc

cfg = wc.get_llm_config()
client = OpenAIChatCompletionClient(model=cfg.model, api_key=cfg.api_key, base_url=cfg.base_url)


@tool
def web_search(query: str) -> str:
    """Search the web for up-to-date information."""
    return wc.web_search(query)


@tool
def word_count(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())


agent = Agent(
    client,
    instructions="You are a helpful assistant. Use your tools when they help answer accurately.",
    name="assistant",
    tools=[web_search, word_count],
)


async def main():
    result = await agent.run("search for the latest langchain version, then tell me how many words your answer is.")
    print(result.text)


if __name__ == "__main__":
    await main()

## 3. `03_agent_with_memory.py`

Microsoft Agent Framework 03 — giving your agent memory
an AgentSession holds the conversation. same session = same memory.

In [ ]:
import asyncio

from agent_framework import Agent
from agent_framework.openai import OpenAIChatCompletionClient

from workshop_common import get_llm_config

cfg = get_llm_config()
client = OpenAIChatCompletionClient(model=cfg.model, api_key=cfg.api_key, base_url=cfg.base_url)
agent = Agent(client, instructions="You are a helpful assistant. Be concise and accurate.", name="assistant")


async def main():
    session = agent.create_session()                    # <- the memory
    r1 = await agent.run("hi! my name is m0h.", session=session)
    print(r1.text)
    r2 = await agent.run("what's my name?", session=session)
    print(r2.text)


if __name__ == "__main__":
    await main()

## 4. `04_agent_with_fallback.py`

Microsoft Agent Framework 04 — the fallback setup

In [ ]:
import asyncio

from agent_framework import Agent
from agent_framework.openai import OpenAIChatCompletionClient

from workshop_common import fallback_order, get_llm_config, ping


def get_client():
    for provider in fallback_order():
        try:
            cfg = get_llm_config(provider)
            ping(cfg)
            print(f"using {provider} ({cfg.model})")
            return OpenAIChatCompletionClient(model=cfg.model, api_key=cfg.api_key, base_url=cfg.base_url)
        except Exception as e:
            print(f"{provider} failed ({str(e)[:80]}); trying the next one")
    raise RuntimeError("every provider failed — check your keys in .env")


agent = Agent(get_client(), instructions="You are a helpful assistant. Be concise.", name="assistant")


async def main():
    print((await agent.run("say hi and tell me which model you are.")).text)


if __name__ == "__main__":
    await main()

## 5. `05_multi_agent.py`

Microsoft Agent Framework 05 — multi-agent with a SEQUENTIAL workflow
researcher -> writer. SequentialBuilder chains agents: each one sees the conversation so far.
(also available: ConcurrentBuilder, GroupChatBuilder, HandoffBuilder, MagenticBuilder)

In [ ]:
import asyncio

from agent_framework import Agent, tool
from agent_framework.orchestrations import SequentialBuilder
from agent_framework.openai import OpenAIChatCompletionClient

import workshop_common as wc

cfg = wc.get_llm_config()
client = OpenAIChatCompletionClient(model=cfg.model, api_key=cfg.api_key, base_url=cfg.base_url)


@tool
def web_search(query: str) -> str:
    """Search the web for up-to-date information."""
    return wc.web_search(query)


@tool
def word_count(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())


researcher = Agent(client, name="researcher", tools=[web_search],
                   instructions="Use web_search to find current facts about the topic. Reply with 3 short bullet facts.")
writer = Agent(client, name="writer", tools=[word_count],
               instructions="Using the researcher's facts, write one friendly paragraph under 60 words. "
                            "Check the length with word_count.")

workflow = SequentialBuilder(participants=[researcher, writer], output_from="all").build()  # show every agent


async def main():
    result = await workflow.run("the latest langchain version")
    for response in result.get_outputs():           # one AgentResponse per agent, in order
        print(f"\n[{response.messages[0].author_name}]\n{response.text}")


if __name__ == "__main__":
    await main()

## 6. `agent.py`

Microsoft Agent Framework — the complete agent (fallback + tools + memory)

In [ ]:
import asyncio

from agent_framework import Agent, tool
from agent_framework.openai import OpenAIChatCompletionClient

import workshop_common as wc


def get_client():
    for provider in wc.fallback_order():
        try:
            cfg = wc.get_llm_config(provider)
            wc.ping(cfg)
            print(f"using {provider} ({cfg.model})")
            return OpenAIChatCompletionClient(model=cfg.model, api_key=cfg.api_key, base_url=cfg.base_url)
        except Exception as e:
            print(f"{provider} failed ({str(e)[:80]}); trying the next one")
    raise RuntimeError("every provider failed — check your keys in .env")


@tool
def web_search(query: str) -> str:
    """Search the web for up-to-date information."""
    return wc.web_search(query)


@tool
def word_count(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())


agent = Agent(
    get_client(),
    name="assistant",
    instructions="You are a helpful assistant. Use your tools when they help answer accurately, and remember the conversation.",
    tools=[web_search, word_count],
)


async def main():
    session = agent.create_session()
    r1 = await agent.run("my name is m0h. search for the latest langchain version, then tell me how many words your "
                         "answer is.", session=session)
    print(r1.text)
    r2 = await agent.run("what's my name?", session=session)
    print(r2.text)


if __name__ == "__main__":
    await main()

---
*SA-LIB AI Workshop 2026 · based on [build-ai-agents-free](https://github.com/Moh4696/build-ai-agents-free) by m0h (MIT)*